# How one paper benchmark is created: K3

Follow [generate_paper_benchmarks.py](../../tools/generate_paper_benchmarks.py)
from a DIMACS graph to **one row containing five methods**. We construct the
Logique circuit in small steps, generate the four Saha–Belletti baselines, then
run the script itself and reconcile every metric.

K3 is a triangle: three vertices, three edges, and three colors. Its six input
bits make the whole predicate small enough to check exhaustively and draw.

**Route:** graph → edge predicate → expression-based XAG synthesis → clean
phase kickback → one Grover iteration → five-method costs → exported artifacts.

This notebook reproduces the paper's historical choices. In particular, it
removes precolors, leaves unused binary color codes unconstrained, and uses a
diffusion reflection that differs from the initial state preparation. The
[graph-coloring tutorial](graph_coloring.ipynb) covers the maintained full
coloring predicate and matching preparation/reflection instead.

## 1. Use the prepared research environment

Complete the [setup guide](../../docs/setup.md), including the native bridge,
comparison and notebook extras, system Graphviz, and the manuscript Qiskit
MCX-depth patch (`tools/patch_qiskit_depth.py`). These cells check the setup;
they do not install packages or compile dependencies.

Run all cells in order with the research kernel, starting from the repository
root. The notebook runner supplies `LOGIQUE_WORKSPACE` and `LOGIQUE_OUTPUT`.
An interactive run creates its own timestamped folder under `dump/`.

In [ ]:
from pathlib import Path
from datetime import datetime
from itertools import count
import importlib.util
import inspect
import json
import os
import shutil

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
SCRIPT = ROOT / "tools/generate_paper_benchmarks.py"
assert SCRIPT.is_file(), "Launch from the checkout root or set LOGIQUE_WORKSPACE."

required_packages = (
    "qiskit",
    "tweedledum",
    "saha_belletti",
    "pandas",
    "matplotlib",
    "graphviz",
    "schemdraw",
    "pylatexenc",
)
missing = [name for name in required_packages if importlib.util.find_spec(name) is None]
assert not missing, f"Missing {missing}. Follow docs/setup.md with research extras."
assert shutil.which("dot"), "Install system Graphviz (the dot executable)."

from qiskit import QuantumCircuit, ClassicalRegister
from logique.paths import native_executable

assert "mcx_decomposition" in inspect.signature(QuantumCircuit.depth).parameters, (
    "Install the manuscript depth patch with tools/patch_qiskit_depth.py, "
    "then restart the kernel."
)
EXECUTABLE = native_executable(
    os.environ.get("LOGIQUE_NATIVE_EXECUTABLE")
    or ROOT / "external/native/boolean_synthesis/build/boolean_synthesis"
)
os.environ["LOGIQUE_NATIVE_EXECUTABLE"] = str(EXECUTABLE)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Markdown, Math, SVG, Image
from dataclasses import replace
from qiskit.circuit.library.data_preparation.state_preparation import (
    UniformSuperpositionGate,
)
from qiskit.quantum_info import Statevector
from logique.coloring import read_dimacs, encode_coloring, edge_coloring_expression
from logique.synthesis import synthesize_expression
from logique.synthesis.saha_belletti import comparison_circuit
from logique.circuits.grover import kickback_grover_circuit
from logique.verification import verify_kickback_oracle
from logique.verification.classical import input_assignments, truth_outputs
from logique.benchmarks.runner import new_run_directory
from logique.visualization.graphs import draw_colored_graph
from logique.visualization.circuits import circuit_window
from logique.visualization.results import plot_results
from logique.visualization import boolean as bv

# Load the existing script's metric helper and generator without running its CLI.
spec = importlib.util.spec_from_file_location("paper_benchmarks", SCRIPT)
paper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(paper)

BENCHMARK = "K3"
if "LOGIQUE_OUTPUT" in os.environ:
    OUT = Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
else:
    # Explicit named paths also work with the original one-argument run helper.
    run_name = f"{datetime.now().astimezone():%m%d%H%M%S}-paper-benchmark-walkthrough"
    for index in count(1):
        candidate = run_name if index == 1 else f"{run_name}-{index}"
        try:
            OUT = new_run_directory(output=ROOT / "dump" / candidate)
            break
        except FileExistsError:
            continue
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})


def show_figure(figure, stem: str) -> None:
    """Display a figure and save raster and zoomable vector versions in this run."""
    figure.savefig(FIGURES / f"{stem}.svg", bbox_inches="tight")
    figure.savefig(FIGURES / f"{stem}.png", dpi=140, bbox_inches="tight")
    display(Image(filename=str(FIGURES / f"{stem}.png")))
    plt.close(figure)


print("Native bridge:", EXECUTABLE)
print("Artifacts:", OUT)


## 2. Read the graph, then remove its precolor

The file declares `n 1 0`: vertex 1 is fixed to color 0. The paper generator
explicitly removes this constraint with `replace(original, precolored={})`.
The gray triangle below is therefore the graph used in the replay.

The drawings use `logique.visualization.graphs.draw_colored_graph()` with a
fixed circular layout. Vertex labels stay one-based, as in the source file.

In [ ]:
input_path = ROOT / "datasets/graphs/benchmarks" / f"{BENCHMARK}.col"
print(input_path.read_text())
original = read_dimacs(input_path)
problem = replace(original, precolored={})
print("Removed precolors:", dict(original.precolored))
print(
    "Replay vertices, edges, colors:", problem.vertices, problem.edges, problem.colors
)

palette = ("#4c78a8", "#f58518", "#54a24b")
for title, instance, stem in (
    ("Source graph: vertex 1 fixed to color 0", original, "source-graph"),
    ("Paper replay: precolor removed", problem, "paper-graph"),
):
    graph = instance.to_networkx()
    for vertex in graph:
        color = instance.precolored.get(vertex)
        graph.nodes[vertex]["color"] = "#dce3eb" if color is None else palette[color]
        graph.nodes[vertex]["color_idx"] = -1 if color is None else color
    draw_colored_graph(
        graph,
        name=str(FIGURES / f"{stem}.png"),
        pos=nx.circular_layout(graph),
        figsize=(4, 3),
        node_size=1600,
    )
    figure = plt.gcf()
    figure.axes[0].set_title(title)
    figure.axes[0].margins(0.20)
    show_figure(figure, stem)


## 3. Turn each edge into a Boolean inequality

Two bits encode each vertex color, least-significant bit first:
$00\mapsto0$, $01\mapsto1$, $10\mapsto2$, $11\mapsto3$.
Color 3 is an unused code for this three-color problem.

For an edge $(u,v)$, the predicate is
$(v_{u,0}\oplus v_{v,0})\lor(v_{u,1}\oplus v_{v,1})$.
AND the three edge predicates to require that adjacent colors differ.
`edge_coloring_expression()` preserves the expression and variable ordering
used by the original generator. It adds neither precolors nor color-range
constraints.

We use a four-color `reference_encoding` only to decode all six-bit inputs and
provide an independent edge-only verifier. This does **not** change the
three-color initial preparation used later.

In [ ]:
expression, variables = edge_coloring_expression(problem)
bits_per_color = (problem.colors - 1).bit_length()
reference_encoding = encode_coloring(replace(problem, colors=1 << bits_per_color))
input_count = reference_encoding.input_count
mapping = pd.DataFrame(
    [
        {
            "vertex": vertex,
            "color bit": bit,
            "input qubit": wire,
            "expression variable": variables[wire],
        }
        for vertex, wires in reference_encoding.input_mapping.items()
        for bit, wire in enumerate(wires)
    ]
)
display(mapping)
print(expression)
assert input_count == 6


In [ ]:
assignments = [reference_encoding.decode(basis) for basis in range(1 << input_count)]
edge_valid = np.array([reference_encoding.problem.is_valid(a) for a in assignments])
three_color_valid = np.array([problem.is_valid(a) for a in assignments])
original_valid = np.array([original.is_valid(a) for a in assignments])
uses_unused_code = np.array(
    [any(c >= problem.colors for c in a.values()) for a in assignments]
)

display(
    pd.DataFrame(
        {
            "predicate": [
                "Paper edge-only",
                "Three colors, no precolor",
                "Original file with precolor",
            ],
            "accepted bitstrings out of 64": [
                int(edge_valid.sum()),
                int(three_color_valid.sum()),
                int(original_valid.sum()),
            ],
        }
    )
)
# Show one ordinary valid coloring and one accepted assignment using color 3.
examples = [
    int(np.flatnonzero(three_color_valid)[0]),
    int(np.flatnonzero(edge_valid & uses_unused_code)[0]),
]
display(
    pd.DataFrame(
        [
            {
                "basis": basis,
                "q5...q0": f"{basis:06b}",
                "vertex colors": assignments[basis],
                "edge-only accepts": bool(edge_valid[basis]),
                "three-color accepts": bool(three_color_valid[basis]),
            }
            for basis in examples
        ]
    )
)
assert (
    int(edge_valid.sum()),
    int(three_color_valid.sum()),
    int(original_valid.sum()),
) == (24, 6, 2)


## 4. Synthesize the expression and inspect the logic

`synthesize_expression()` compiles the expression directly through tweedledum
and applies specialized XAG synthesis. It also exports Verilog and obtains
native Boolean networks for reference checks and visualization.

The **reference networks** below describe the same Boolean function. They are
reimported by the native bridge and may be refactored; their node IDs are not
a gate-by-gate trace of the expression-tree circuit. The quantum diagrams in
the next section show the actual synthesis output.

We reuse `latex_blocks()` for factored equations, `graphviz_graph()` for a
layered SVG, and `schematic_figure()` for connected digital gates. Here
$x_i$ corresponds to input qubit $i$ in the mapping above. Complemented edges
are inversions; a drawn crossing is connected only where marked by a dot.

In [ ]:
synthesis = synthesize_expression(expression, variables, executable=EXECUTABLE)
source_network = synthesis.metadata["source_network"]
reference_network = synthesis.metadata["network"]
(OUT / "K3_expression.v").write_text(synthesis.metadata["source_verilog"])
print(synthesis.metadata["source_verilog"])

display(
    pd.DataFrame(
        [
            {"representation": label, **bv.graph_stats(network)}
            for label, network in (
                ("Native source reference", source_network),
                ("Native mapped reference", reference_network),
            )
        ]
    )
)
# Check that both the exported reference and the displayed equations match edges.
input_bits = input_assignments(input_count)
np.testing.assert_array_equal(truth_outputs(source_network, input_bits)[0], edge_valid)
np.testing.assert_array_equal(
    bv.evaluate_gate_rows(reference_network, input_bits)[0], edge_valid
)
for block in bv.latex_blocks(reference_network, rows_per_block=6):
    display(Math(block))


In [ ]:
for label, network, stem in (
    (
        "Native source reference for the edge predicate",
        source_network,
        "source-reference-network",
    ),
    (
        "Native mapped reference for the edge predicate",
        reference_network,
        "mapped-reference-network",
    ),
):
    dot = bv.graphviz_graph(network, label)
    (FIGURES / f"{stem}.dot").write_text(dot.source)
    svg = dot.pipe(format="svg")
    (FIGURES / f"{stem}.svg").write_bytes(svg)
    display(SVG(data=svg))

show_figure(bv.schematic_figure(reference_network), "edge-predicate-schematic")


## 5. Inspect and verify the actual quantum computation

The raw circuit has data inputs, one output target, and workspace. Keep the
phases of controlled $R_x(\pm\pi)$ gates: they cannot be replaced by Toffolis
just because their computational-basis actions look similar.

`circuit_window()` shows a labeled excerpt, followed by the complete raw
circuit. An excerpt is a reading aid, not an executable sub-oracle.

In [ ]:
inputs = synthesis.metadata["input_qubits"]
outputs = synthesis.metadata["output_qubits"]
workspace = sorted(
    set(range(synthesis.circuit.num_qubits)) - set(inputs) - set(outputs)
)
display(
    pd.DataFrame(
        [
            {"role": "Data", "qubits": inputs},
            {"role": "Output target", "qubits": outputs},
            {"role": "Workspace", "qubits": workspace},
        ]
    )
)
print("Raw allocated qubits:", synthesis.circuit.num_qubits)
print("Raw instructions:", len(synthesis.circuit))
display(
    pd.Series(dict(synthesis.circuit.count_ops()), name="operation count").to_frame()
)
show_figure(circuit_window(synthesis, start=0, count=16), "raw-circuit-excerpt")
show_figure(
    synthesis.circuit.draw("mpl", fold=24, idle_wires=False), "raw-circuit-full"
)


### Why check both target values?

Before phase kickback, require the raw computation to implement
$|x\rangle|y\rangle|0\rangle\mapsto
|x\rangle|y\oplus f(x)\rangle|0\rangle$ for **both** $y=0$ and $y=1$,
with no extra input-dependent phase. By linearity, starting the target in
$|-\rangle$ then applies $(-1)^{f(x)}$ and restores the workspace.

`verify_kickback_oracle()` checks all 64 inputs, both target bits, phase,
input preservation, and workspace restoration against the independent graph
predicate. This establishes the Logique oracle contract; it does not verify
the historical baseline circuits.

In [ ]:
verification = verify_kickback_oracle(
    synthesis, encoding=reference_encoding, sampled=False, seed=7
)
display(pd.Series(verification, name="verification").to_frame())
assert verification["status"] == "exhaustive"
assert verification["assignments"] == 64
assert verification["marked_assignments"] == 24
assert verification["target_initializations"] == [0, 1]
assert all(
    verification[key]
    for key in ("phase_exact", "inputs_restored", "workspace_restored")
)
(OUT / "kickback_verification.json").write_text(
    json.dumps(verification, indent=2) + "\n"
)


## 6. Prepare three colors, then apply historical diffusion

Each vertex starts in
$|s_3\rangle=(|00\rangle+|01\rangle+|10\rangle)/\sqrt3$,
so the data register initially contains 27 equally likely assignments, with
no unused code 3. Use the same `UniformSuperpositionGate(3)` decomposition as
the generator: substituting another equivalent preparation could change costs.

For replay, diffusion reflects about $|s_{64}\rangle=H^{\otimes6}|0\rangle$,
not about $|s_3\rangle^{\otimes3}$. The historical helper implements
$I-2|s_{64}\rangle\langle s_{64}|$. Its reflection can introduce amplitude
on unused codes. The two preparations below make that distinction visible.

In [ ]:
vertex_preparation = QuantumCircuit(bits_per_color)
vertex_preparation.append(
    UniformSuperpositionGate(problem.colors), range(bits_per_color)
)
vertex_preparation = vertex_preparation.decompose()
preparation = QuantumCircuit(input_count)
for start in range(0, input_count, bits_per_color):
    preparation.compose(
        vertex_preparation, range(start, start + bits_per_color), inplace=True
    )

reflection_preparation = QuantumCircuit(input_count)
reflection_preparation.h(range(input_count))
show_figure(preparation.draw("mpl", fold=24), "valid-color-preparation")
show_figure(reflection_preparation.draw("mpl", fold=24), "reflection-preparation")

vcgc = kickback_grover_circuit(
    synthesis,
    preparation=preparation,
    reflection_preparation=reflection_preparation,
    iterations=1,
)
show_figure(vcgc.draw("mpl", fold=24, idle_wires=False), "one-grover-iteration")


The full circuit initializes the output in $|-\rangle$, applies the raw XAG
once, then diffuses the data. No extra inverse of the XAG is added: its clean
XOR action was checked above.

For this 11-qubit example we can simulate the complete state locally and
sum probabilities over the output and workspace. The groups below separate
proper three-colorings, color conflicts using only valid codes, and all
assignments containing the unused code. This is an exact noiseless illustration
of one historical iteration, not an optimized iteration count.

In [ ]:
assert vcgc.num_qubits <= 16, "Keep dense statevector demonstrations small."
before = Statevector.from_instruction(preparation).probabilities()
final_state = Statevector.from_instruction(vcgc)
after = final_state.probabilities(inputs)
np.testing.assert_allclose(before.sum(), 1, atol=1e-9)
np.testing.assert_allclose(after.sum(), 1, atol=1e-9)
np.testing.assert_allclose(before[uses_unused_code], 0, atol=1e-9)
np.testing.assert_allclose(final_state.probabilities(workspace)[0], 1, atol=1e-9)
assert after[uses_unused_code].sum() > 0

groups = {
    "Proper three-coloring": three_color_valid,
    "Valid codes, edge conflict": ~uses_unused_code & ~three_color_valid,
    "Contains unused code 3": uses_unused_code,
}
probabilities = pd.DataFrame(
    [
        {
            "assignment group": label,
            "bitstrings": int(mask.sum()),
            "Initial": float(before[mask].sum()),
            "After one iteration": float(after[mask].sum()),
        }
        for label, mask in groups.items()
    ]
).set_index("assignment group")
display(probabilities)
probabilities.to_csv(OUT / "probability_groups.csv")
figure, ax = plt.subplots(figsize=(9, 4))
probabilities[["Initial", "After one iteration"]].plot.bar(ax=ax, rot=0)
ax.set_ylabel("Total input probability")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.set_title("K3: initial valid-color state and historical Grover iteration")
figure.tight_layout()
show_figure(figure, "probability-groups")


## 7. Construct the other four methods

The generator uses `original`, `minimal`, `simple`, and `balanced` variants
of the Saha–Belletti dependency, each with one Grover iteration and a
Hadamard-prepared data register. Its adapter preserves one-based graph labels
with `preserve_labels=True`, including their mismatch with the dependency's
indexing, and retains final measurements with `measure=True`.

These settings reproduce historical circuits and costs. The comparison does
not establish equivalent preparations or correctness of those baseline
circuits. We retain the baseline circuits for export, show one representative
diagram here, and save all four diagrams for inspection.

In [ ]:
methods = ("vcgc", *(f"sb_{variant}" for variant in paper.VARIANTS))
circuits = {"vcgc": vcgc}
for variant in paper.VARIANTS:
    method = f"sb_{variant}"
    circuits[method] = comparison_circuit(
        problem, variant, iterations=1, preserve_labels=True, measure=True
    )
    figure = circuits[method].draw("mpl", fold=24, idle_wires=False)
    figure.savefig(FIGURES / f"{method}-circuit.svg", bbox_inches="tight")
    if variant == "original":
        show_figure(figure, f"{method}-circuit")
    else:
        plt.close(figure)

display(
    pd.DataFrame(
        [
            {
                "method": method,
                "allocated qubits": circuit.num_qubits,
                "classical bits": circuit.num_clbits,
                "measurements": int(circuit.count_ops().get("measure", 0)),
            }
            for method, circuit in circuits.items()
        ]
    )
)
assert len(circuits) == 5


## 8. Count exactly what the paper counts

Reuse the script's `paper_metrics()` rather than substituting ordinary
Qiskit resource counts.

| Reported metric | Historical definition |
|---|---|
| Width | Allocated circuit qubits plus $\sum\max(0,n-2)$ over `mcx` instructions except the last, where $n$ counts controls. The last is assumed to be diffusion. This allowance is not allocated workspace. |
| Depth | `circuit.depth(mcx_decomposition=True)`: the local patch weights an $n$-controlled X by $\lceil\log_2 n\rceil$ for $n\ge2$. This is an analytical estimate, not a synthesized hardware or T-depth. |
| Gates | `len(circuit)`, including barriers and measurements. |

The historical table leaves Logique unmeasured while measuring all baselines.
For the second table, add one final barrier and six data measurements to a
**copy** of Logique. The other methods already have these measurements.

In [ ]:
measured_vcgc = vcgc.copy()
measured_vcgc.add_register(ClassicalRegister(input_count, "creg"))
measured_vcgc.barrier()
measured_vcgc.measure(range(input_count), range(input_count))
measured_circuits = {**circuits, "vcgc": measured_vcgc}

historical_costs = {method: paper.paper_metrics(circuits[method]) for method in methods}
measured_costs = {
    method: paper.paper_metrics(measured_circuits[method]) for method in methods
}
historical_table = pd.DataFrame.from_dict(historical_costs, orient="index")
measured_table = pd.DataFrame.from_dict(measured_costs, orient="index")
display(Markdown("**Historical accounting**"))
display(historical_table)
display(Markdown("**All methods measured**"))
display(measured_table)
display(Markdown("**Change from measuring Logique**"))
display(measured_table - historical_table)
assert measured_costs["vcgc"]["gates"] == historical_costs["vcgc"]["gates"] + 7
assert measured_costs["vcgc"]["depth"] == historical_costs["vcgc"]["depth"] + 1
assert measured_costs["vcgc"]["width"] == historical_costs["vcgc"]["width"]
for method in methods[1:]:
    assert measured_costs[method] == historical_costs[method]

# Inspect the difference between allocated width and the MCX allowance.
display(
    pd.DataFrame(
        [
            {
                "method": method,
                "allocated qubits": circuits[method].num_qubits,
                "MCX allowance": historical_costs[method]["width"]
                - circuits[method].num_qubits,
                "historical width": historical_costs[method]["width"],
                "ordinary Qiskit depth": circuits[method].depth(),
                "weighted depth estimate": historical_costs[method]["depth"],
            }
            for method in methods
        ]
    )
)


## 9. Run the generator for K3 and reconcile the files

Now call `generate(graphs=["K3"])` from the actual script. It synthesizes afresh,
verifies Logique, writes six QASM circuits and the predicate Verilog, records
provenance, and compares its 15 historical metrics against the archived K3
row. Archived numbers supply comparison targets; they are not used to populate
new costs.

Keep this generator run in `paper-replay/` so its `run.json` does not overwrite
the notebook runner's manifest. The assertions compare our stepwise construction
with both freshly generated tables and check exact historical agreement in the
pinned environment. A mismatch remains visible as a failed assertion and in
`comparison.csv`.

The generator refuses to overwrite an existing directory. To repeat this cell,
start a new notebook run (or rerun setup interactively to allocate a new folder).

In [ ]:
replay = paper.generate(output=OUT / "paper-replay", graphs=[BENCHMARK])
historical_csv = pd.read_csv(replay / "benchmark_results.csv").set_index("benchmark")
measured_csv = pd.read_csv(replay / "measurement_consistent_results.csv").set_index(
    "benchmark"
)
for method in methods:
    for metric in ("width", "depth", "gates"):
        assert (
            historical_csv.loc[BENCHMARK, f"{method}_{metric}"]
            == historical_costs[method][metric]
        )
        assert (
            measured_csv.loc[BENCHMARK, f"{method}_{metric}"]
            == measured_costs[method][metric]
        )

comparison = pd.read_csv(replay / "comparison.csv")
display(comparison)
assert len(comparison) == 15
np.testing.assert_array_equal(
    comparison["delta"], comparison["new"] - comparison["old"]
)
assert comparison[
    "exact"
].all(), "Historical mismatch: inspect comparison.csv and dependency versions."
manifest = json.loads((replay / "run.json").read_text())
assert manifest["status"] == "complete"
assert manifest["cases"][0]["workspace_restored"]
assert len(list(replay.glob("*.qasm"))) == 6
print(f"{int(comparison['exact'].sum())}/15 historical metrics match.")


## 10. Visualize the five-method costs

`logique.visualization.results.plot_results()` reads the wide paper CSVs through
the existing result loader. It maps the `vcgc` prefix to method `logique` and
`width` to plotting metric `qubits`. We replace the default axis labels to
state the historical allowance and weighted-depth model precisely.

Each metric is plotted separately for historical and measurement-consistent
accounting. These figures compare costs under the recorded replay choices;
method rankings alone do not compare coloring success probabilities.

In [ ]:
method_labels = {
    "logique": "Logique (XAG)",
    "sb_original": "Saha original",
    "sb_minimal": "Belletti minimal",
    "sb_simple": "Belletti simple",
    "sb_balanced": "Belletti balanced",
}
for accounting, filename in (
    ("historical", "benchmark_results.csv"),
    ("all-measured", "measurement_consistent_results.csv"),
):
    for metric, ylabel in (
        ("qubits", "Width with historical MCX allowance"),
        ("depth", "MCX tree-weighted depth estimate"),
        ("gates", "Instruction count (barriers and measurements included)"),
    ):
        figure = plot_results(
            replay / filename, metric=metric, method_labels=method_labels
        )
        ax = figure.axes[0]
        ax.set_ylabel(ylabel)
        ax.set_title(f"K3 — {accounting} accounting")
        ax.tick_params(axis="x", labelrotation=0)
        show_figure(figure, f"{accounting}-{metric}")


## 11. Inspect artifacts and provenance

Open the executed notebook to retain inline visuals, the SVG files to zoom
into circuits, or the QASM files to inspect the generated instructions.
`paper-replay/run.json` records package versions, checkout revision, generator
hash, depth model, replay conventions, and verification results. The parent
`run.json`, when using the notebook runner, records notebook execution separately.

The source notebook stays unexecuted in version control. All generated data and
figures remain together in this run folder.

In [ ]:
display(
    pd.Series(
        {
            "depth model": manifest["depth_model"],
            "precolors": manifest["precolors"],
            "encoding": manifest["encoding"],
            "preparation": manifest["preparation"],
            "oracle": manifest["oracle"],
            "baseline labels": manifest["baseline_labels"],
        },
        name="replay conventions",
    ).to_frame()
)
display(pd.Series(manifest["versions"], name="version").to_frame())

link_base = OUT if "LOGIQUE_OUTPUT" in os.environ else ROOT / "experiments/tutorials"
for label, path in (
    ("Run provenance", replay / "run.json"),
    ("Historical metrics", replay / "benchmark_results.csv"),
    ("All-measured metrics", replay / "measurement_consistent_results.csv"),
    ("Archived-versus-current comparison", replay / "comparison.csv"),
    ("Logique Grover circuit", replay / "K3_vcgc.qasm"),
    ("Measured Logique circuit", replay / "K3_vcgc_measured.qasm"),
    ("Digital logic schematic", FIGURES / "edge-predicate-schematic.svg"),
    ("Full Grover diagram", FIGURES / "one-grover-iteration.svg"),
):
    assert path.is_file()
    relative_link = os.path.relpath(path, link_base)
    display(Markdown(f"[{label}]({relative_link})"))

print("All files:")
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT))


To reproduce the walkthrough from the repository root:

```bash
uv run --no-sync logique notebook run experiments/tutorials/paper_benchmark_walkthrough.ipynb --offline
```

Questions to explore after this replay: Which costs come from state preparation,
oracle computation, diffusion, and measurement? How does enforcing the full
three-color predicate change the oracle? How does reflecting about the initial
state change the probabilities? Use the separate full-predicate tutorial for
those changes; changing these choices would define a different benchmark from
the archived paper row.